In [ ]:
import os, sys, json, ast, argparse, shutil, time, random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm

!pip install -q rouge-score nltk gdown pycocoevalcap 2>/dev/null

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


SEED = 9223  
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

In [ ]:
import pandas as pd
from pathlib import Path


CHUNK01_DIR  = Path('/kaggle/input/datasets/muhammadaneeq786/mimic-cxr-chunk-01-tar')
MANIFEST_DIR = Path('/kaggle/input/datasets/muhammadaneeq786/mimic-cxr-master-manifest')

OUTPUT_DIR = "/kaggle/working/outputs"
R2GEN_DIR  = "/kaggle/working/R2Gen"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for _d, _n in [(CHUNK01_DIR, 'CHUNK01_DIR'), (MANIFEST_DIR, 'MANIFEST_DIR')]:
    assert _d.exists(), f"\n❌ MISSING DATASET: {_n} not found at {_d}\n   → Attach the Kaggle dataset and re-run."
    print(f"✅ {_n}: {_d}")


def _find_csv(filename, *search_dirs):
    for d in search_dirs:
        p = Path(d) / filename
        if p.exists():
            return p
        hits = list(Path(d).rglob(filename))
        if hits:
            print(f"  ℹ  {filename} found at: {hits[0]}")
            return hits[0]
    return None


metadata_csv  = _find_csv('mimic-cxr-2.0.0-metadata.csv', CHUNK01_DIR, MANIFEST_DIR)
split_csv     = _find_csv('mimic-cxr-2.0.0-split.csv',    CHUNK01_DIR, MANIFEST_DIR)
chexpert_csv  = _find_csv('mimic-cxr-2.0.0-chexpert.csv', CHUNK01_DIR, MANIFEST_DIR)
chunk01_man   = _find_csv('manifest_chunk_01.csv',          CHUNK01_DIR)
master_csv    = _find_csv('master_manifest.csv',            MANIFEST_DIR)
train_man_csv = _find_csv('manifest_train_50k_exact.csv',   MANIFEST_DIR)
val_man_csv   = _find_csv('manifest_val_full.csv',          MANIFEST_DIR)
test_man_csv  = _find_csv('manifest_test_full.csv',         MANIFEST_DIR)

print("\nChecking required files:")
for name, path in [('metadata', metadata_csv), ('split', split_csv),
                   ('chexpert', chexpert_csv),  ('chunk01_manifest', chunk01_man),
                   ('master_manifest', master_csv)]:
    assert path is not None, f"Required CSV not found: {name}.csv"
    print(f"  ✅ {name}: {path}")


print("\nLoading CSVs...")
metadata_df = pd.read_csv(metadata_csv)
split_df    = pd.read_csv(split_csv)
chexpert_df = pd.read_csv(chexpert_csv)
chunk01_df  = pd.read_csv(chunk01_man)
master_df   = pd.read_csv(master_csv)

def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns: return c
    low = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in low: return low[c.lower()]
    return df.columns[0]

DCOL_META  = _find_col(metadata_df, ['dicom_id', 'DicomId', 'DICOM_ID'])
DCOL_SPLIT = _find_col(split_df,    ['dicom_id', 'DicomId'])
DCOL_C01   = _find_col(chunk01_df,  ['dicom_id', 'DicomId', 'dicom'])
SCOL_MAST  = _find_col(master_df,   ['study_id', 'StudyId', 'study'])

print(f"  Columns → meta:{DCOL_META!r}  split:{DCOL_SPLIT!r}  chunk01:{DCOL_C01!r}  master:{SCOL_MAST!r}")


chunk01_dicoms = set(chunk01_df[DCOL_C01].astype(str).str.strip())
metadata_df[DCOL_META] = metadata_df[DCOL_META].astype(str).str.strip()

frontal_df = metadata_df[
    metadata_df['ViewPosition'].isin(['PA', 'AP']) &
    metadata_df[DCOL_META].isin(chunk01_dicoms)
].copy().rename(columns={DCOL_META: 'dicom_id'})

frontal_df['image_filename'] = frontal_df['dicom_id'] + '.jpg'
IMAGE_DIR = CHUNK01_DIR  


_sample_n = min(200, len(frontal_df))
_n_found = sum((IMAGE_DIR / f).exists() for f in frontal_df['image_filename'].iloc[:_sample_n])
if _n_found == 0:
    _jpgs = list(CHUNK01_DIR.rglob('*.jpg'))
    if _jpgs:
        IMAGE_DIR = _jpgs[0].parent
        _n_found = sum((IMAGE_DIR / f).exists() for f in frontal_df['image_filename'].iloc[:_sample_n])
        print(f"  ℹ  Auto-detected IMAGE_DIR: {IMAGE_DIR}")
print(f"Image check ({_sample_n} sample): {_n_found}/{_sample_n} found  ({_n_found/_sample_n*100:.0f}%)")


split_df[DCOL_SPLIT] = split_df[DCOL_SPLIT].astype(str).str.strip()
frontal_df = frontal_df.merge(
    split_df[[DCOL_SPLIT, 'split']].rename(columns={DCOL_SPLIT: 'dicom_id'}),
    on='dicom_id', how='inner'
)


frontal_df['_vp'] = frontal_df['ViewPosition'].map({'PA': 0, 'AP': 1}).fillna(2)
frontal_df = frontal_df.sort_values(['study_id', '_vp'])
frontal_df['_rank'] = frontal_df.groupby('study_id').cumcount()

study_df = frontal_df.drop_duplicates('study_id', keep='first').copy()
_sec = (frontal_df[frontal_df['_rank'] == 1][['study_id', 'dicom_id']]
        .rename(columns={'dicom_id': 'secondary_dicom_id'}))
study_df = study_df.merge(_sec, on='study_id', how='left')
study_df.drop(columns=['_vp', '_rank'], errors='ignore', inplace=True)



chexpert_df['study_id'] = (chexpert_df['study_id'].astype(str).str.strip()
                           .str.replace(r'^s', '', regex=True))
study_df['study_id']    = study_df['study_id'].astype(str).str.strip()
study_df = study_df.merge(
    chexpert_df.drop(columns=['subject_id'], errors='ignore'),
    on='study_id', how='left'
)
CHEXPERT_COLS = [c for c in chexpert_df.columns if c not in ('subject_id', 'study_id')]


_chex_joined = study_df[CHEXPERT_COLS[0]].notna().sum() if CHEXPERT_COLS else 0
assert _chex_joined > 0, (
    f"STOP: CheXpert merge produced 0 non-NaN rows.\n"
    f"  study_df sample IDs: {study_df['study_id'].iloc[:3].tolist()}\n"
    f"  chexpert sample IDs (after strip): {chexpert_df['study_id'].iloc[:3].tolist()}\n"
    "  Study ID format mismatch — check both CSVs and re-run."
)
print(f"  CheXpert labels joined: {_chex_joined:,} rows have labels")




master_df[SCOL_MAST] = (master_df[SCOL_MAST].astype(str).str.strip()
                        .str.replace(r'^s', '', regex=True))
study_df = study_df[study_df['study_id'].isin(set(master_df[SCOL_MAST]))].copy()


train_df    = study_df[study_df['split'] == 'train'   ].reset_index(drop=True)
val_subset  = study_df[study_df['split'] == 'validate'].reset_index(drop=True)
test_subset = study_df[study_df['split'] == 'test'    ].reset_index(drop=True)

assert len(test_subset) > 0, (
    f"STOP: test_subset is empty after all filtering.\n"
    f"  study_df total: {len(study_df)}\n"
    f"  master_manifest rows: {len(master_df)}\n"
    f"  frontal_df rows: {len(frontal_df)}\n"
    "  Check that chunk01, metadata, split, and master_manifest are consistent."
)


def get_best_image(row):
    
    fn = row.get('image_filename') if hasattr(row, 'get') else getattr(row, 'image_filename', None)
    if fn and str(fn) not in ('', 'nan', 'None'):
        return str(fn)
    did = row.get('dicom_id') if hasattr(row, 'get') else getattr(row, 'dicom_id', None)
    return f"{did}.jpg" if did and str(did) not in ('', 'nan', 'None') else None

def get_report_text(row):
    
    return ""


train_pts = set(train_df['subject_id'].unique())
val_pts   = set(val_subset['subject_id'].unique())
test_pts  = set(test_subset['subject_id'].unique())
tv_ol = train_pts & val_pts
tt_ol = train_pts & test_pts
SPLIT_INTEGRITY = "patient_overlap_detected" if (tv_ol or tt_ol) else "verified_patient_level"

print(f"\n{'='*60}")
print(f"MIMIC-CXR Data Loaded  (Chunk 01 — same dataset as Module 1)")
print(f"{'='*60}")
print(f"  Train    : {len(train_df):>6,}")
print(f"  Validate : {len(val_subset):>6,}")
print(f"  Test     : {len(test_subset):>6,}")
print(f"  CheXpert : {len(CHEXPERT_COLS)} pathology columns")
print(f"  IMAGE_DIR: {IMAGE_DIR}")
print(f"  Split integrity: {SPLIT_INTEGRITY}")
print(f"  NOTE: No raw report text in this dataset — CheXpert labels used for evaluation")

_s = test_subset.iloc[0]
_img = get_best_image(_s)
print(f"\nTest sample: study={_s['study_id']}, img={_img}, "
      f"exists={( IMAGE_DIR / _img).exists() if _img else False}")
print(f"✅ Data loaded")

In [ ]:
import warnings
import gdown


ANN_PATH = "/kaggle/input/datasets/muhammadaneeq786/annotation/annotation.json"
print(f"✅ Using pre-uploaded annotation.json: {ANN_PATH}")
assert os.path.exists(ANN_PATH), (
    f"Annotation file not found at: {ANN_PATH}\n"
    "Ensure the 'annotation' Kaggle dataset is attached to this notebook."
)


if not os.path.exists(R2GEN_DIR):
    !git clone https://github.com/cuhksz-nlp/R2Gen.git {R2GEN_DIR}
    print("✅ R2Gen cloned")
else:
    print("✅ R2Gen already exists")


!pip install -q progress 2>/dev/null

sys.path.insert(0, R2GEN_DIR)
from modules.tokenizers import Tokenizer
from models.r2gen import R2GenModel
print("✅ R2Gen modules imported")


CHECKPOINT = "/kaggle/input/datasets/muhammadaneeq786/model-mimic-cxr/model_mimic_cxr.pth"
print(f"✅ Using pre-uploaded checkpoint: {CHECKPOINT}")
assert os.path.exists(CHECKPOINT), (
    f"Checkpoint not found at: {CHECKPOINT}\n"
    "Ensure the 'model-mimic-cxr' Kaggle dataset is attached to this notebook."
)


args = argparse.Namespace(
    
    image_dir=str(IMAGE_DIR),
    ann_path=ANN_PATH,
    dataset_name='mimic_cxr',        
    max_seq_length=100,               
    threshold=10,                     
    
    visual_extractor='resnet101',
    visual_extractor_pretrained=True,
    
    d_model=512,
    d_ff=512,
    d_vf=2048,
    num_heads=8,
    num_layers=3,
    dropout=0.1,
    
    logit_layers=1,
    use_bn=0,
    drop_prob_lm=0.5,
    bos_idx=0,
    eos_idx=0,
    pad_idx=0,
    
    rm_num_slots=3,
    rm_num_heads=8,
    rm_d_model=512,
    
    sample_method='beam_search',
    beam_size=3,
    temperature=1.0,
    sample_n=1,
    group_size=1,
    output_logsoftmax=1,
    decoding_constraint=0,
    block_trigrams=1,
    
    seed=SEED,
    n_gpu=1,
    num_workers=2,
    batch_size=16,
)


tokenizer = Tokenizer(args)
vocab_size = len(tokenizer.token2idx)
print(f"\nTokenizer vocab: {vocab_size} tokens (+ 1 pad = {vocab_size + 1} embedding rows)")


if not hasattr(tokenizer, 'clean_report'):
    import re as _re_tok
    def _clean_report_mimic_cxr(report):
        report_cleaner = lambda t: (t.replace('..', '.').replace('..', '.').replace('..', '.')
                                    .replace('1. ', ' ').replace('. 2. ', ' ').replace('. 3. ', ' ')
                                    .replace('. 4. ', ' ').replace('. 5. ', ' ')
                                    .replace(' 2. ', ' ').replace(' 3. ', ' ')
                                    .replace(' 4. ', ' ').replace(' 5. ', ' ')
                                    .strip().lower().split('. '))
        sent_cleaner = lambda t: _re_tok.sub(r'[.,?;*!%^&_+():\-\[\]{}]', '',
                                              t.replace('"', '').replace('/', '')
                                               .replace('\\', '').replace("'", '').strip().lower())
        tokens = [sent_cleaner(sent) for sent in report_cleaner(report) if sent_cleaner(sent) != ['']]
        report = ' . '.join(tokens)
        return report
    tokenizer.clean_report = _clean_report_mimic_cxr
    print("✅ clean_report monkey-patched (older R2Gen version detected)")



with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
ckpt_state = ckpt.get('state_dict', ckpt)

ckpt_vocab = None
for key in sorted(ckpt_state.keys()):
    if 'tgt_embed' in key and 'lut' in key:
        ckpt_vocab = ckpt_state[key].shape[0]
        print(f"Checkpoint vocab: {ckpt_vocab} (from {key})")
        break

if ckpt_vocab and vocab_size + 1 == ckpt_vocab:
    print("✅ PERFECT MATCH! Tokenizer vocab aligns with checkpoint.")
elif ckpt_vocab:
    print(f"⚠️ MISMATCH: tokenizer={vocab_size + 1}, checkpoint={ckpt_vocab}")
    print("   Root cause: annotation.json was built with a different threshold.")
    print("   Verify that ANN_PATH points to the original R2Gen MIMIC-CXR annotation.json.")


class TokenizerWrapper:
    def __init__(self, tok, target):
        self.tok = tok
        self.idx2token = tok.idx2token
        self.token2idx = tok.token2idx
        self._target = target
    def get_vocab_size(self):
        return self._target
    def __len__(self):
        return self._target
    def __call__(self, report):
        return self.tok(report)
    def decode(self, ids):
        return self.tok.decode(ids)
    def decode_batch(self, ids_batch):
        return self.tok.decode_batch(ids_batch)
    def clean_report(self, report):
        return self.tok.clean_report(report)

wrapped_tok = TokenizerWrapper(tokenizer, ckpt_vocab if ckpt_vocab else vocab_size + 1)
model = R2GenModel(args, wrapped_tok)

try:
    model.load_state_dict(ckpt_state, strict=True)
    print("✅ All checkpoint weights loaded (strict=True)")
except RuntimeError as e:
    
    
    raise RuntimeError(
        f"Checkpoint load failed. This is almost always a threshold mismatch.\n"
        f"Tokenizer vocab+1={vocab_size+1}, Checkpoint vocab={ckpt_vocab}.\n"
        f"Check that ANN_PATH uses the original R2Gen annotation.json (threshold=10).\n"
        f"Original error: {e}"
    ) from e

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(DEVICE).eval()
print(f"✅ R2Gen MIMIC-CXR model on {DEVICE} ({sum(p.numel() for p in model.parameters()):,} params)")

In [ ]:
from PIL import Image
import torchvision.transforms as transforms

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

def load_study_image(row):
    
    img_name = get_best_image(row)
    if img_name and (IMAGE_DIR / img_name).exists():
        try:
            img = Image.open(IMAGE_DIR / img_name).convert('RGB')
            return eval_transform(img)  
        except Exception:
            pass
    return torch.zeros(3, 224, 224)  


test_row = test_subset.iloc[0]
images = load_study_image(test_row).unsqueeze(0).to(DEVICE)  

with torch.no_grad():
    output = model(images, mode='sample')

generated = tokenizer.decode_batch(output.cpu().numpy())[0]

print(f"Study:     {test_row.get('subject_id', 'N/A')}")
print(f"GT:        {get_report_text(test_row)[:150]}...")
print(f"Generated: {generated}")



medical = [
    'heart', 'cardiac', 'cardiomegaly', 'cardiomediastinal',
    'lung', 'lungs', 'pulmonary', 'bilateral',
    'effusion', 'opacity', 'consolidation', 'atelectasis', 'pneumothorax',
    'edema', 'nodule', 'mass', 'mediastinal',
    'tube', 'catheter', 'line', 'pacemaker', 'nasogastric', 'endotracheal',
    'chest', 'normal', 'clear', 'unchanged', 'stable', 'unremarkable',
    'evidence', 'radiograph', 'fracture', 'pleural',
]
found = [t for t in medical if t in generated.lower()]
quality = '✅ Real medical report!' if len(found) >= 2 else '⚠️ Check output — very few medical terms'
print(f"\n{quality}")
print(f"   Terms found ({len(found)}): {found}")

In [ ]:
print(f"Generating reports for {len(test_subset)} test samples...")
start = time.time()

BATCH_SIZE = 16
test_ids = []
test_gts = []
test_reports = []
test_img_paths = []

for batch_start in tqdm(range(0, len(test_subset), BATCH_SIZE), desc="Generating"):
    batch_df = test_subset.iloc[batch_start : batch_start + BATCH_SIZE]

    batch_imgs = []
    for _, row in batch_df.iterrows():
        batch_imgs.append(load_study_image(row))
        test_ids.append(str(row['study_id']))
        test_gts.append(get_report_text(row))
        test_img_paths.append(get_best_image(row) or "")

    images = torch.stack(batch_imgs).to(DEVICE)  

    with torch.no_grad():
        output = model(images, mode='sample')
        reports = tokenizer.decode_batch(output.cpu().numpy())
        test_reports.extend([r.strip() if r else "" for r in reports])

elapsed = time.time() - start
print(f"\n✅ Generated {len(test_reports)} reports in {elapsed:.1f}s ({elapsed/len(test_reports)*1000:.1f}ms each)")


baseline_df = pd.DataFrame({
    'study_id': test_ids,
    'report': test_reports,
    'ground_truth': test_gts,
    'image_path': test_img_paths,
})

baseline_path = os.path.join(OUTPUT_DIR, 'predictions_baseline.csv')
baseline_df.to_csv(baseline_path, index=False)
print(f"✅ Saved: {baseline_path}")


unique = baseline_df['report'].nunique()
print(f"Unique reports: {unique}/{len(baseline_df)}")

print("\n--- Sample outputs ---")
for i in range(min(3, len(baseline_df))):
    r = baseline_df.iloc[i]
    print(f"\n[{i+1}] {r['study_id']}")
    print(f"  GEN: {r['report'][:120]}")
    print(f"  GT:  {r['ground_truth'][:120]}")

In [ ]:
CHEXPERT_REPORT_KEYWORDS = {
    'Atelectasis':                ['atelectasis', 'atelectatic'],
    'Cardiomegaly':               ['cardiomegaly', 'cardiac', 'enlarged', 'cardiomediastinal'],
    'Consolidation':              ['consolidation', 'consolidative'],
    'Edema':                      ['edema', 'edematous', 'interstitial'],
    'Enlarged Cardiomediastinum': ['mediastinal', 'mediastinum', 'widened'],
    'Fracture':                   ['fracture', 'fractured'],
    'Lung Lesion':                ['lesion', 'nodule', 'mass'],
    'Lung Opacity':               ['opacity', 'opaque', 'opacification'],
    'Pleural Effusion':           ['effusion', 'pleural'],
    'Pleural Other':              ['thickening', 'pleural'],
    'Pneumonia':                  ['pneumonia', 'pneumonic', 'infection'],
    'Pneumothorax':               ['pneumothorax'],
    'Support Devices':            ['catheter', 'tube', 'line', 'device', 'pacemaker'],
}


assert len(baseline_df) == len(test_subset), (
    f"Length mismatch: baseline_df={len(baseline_df)}, test_subset={len(test_subset)}"
)
eval_df = test_subset.copy().reset_index(drop=True)
eval_df['generated_report'] = baseline_df['report'].values

results_by_pathology = {}
for entity, keywords in CHEXPERT_REPORT_KEYWORDS.items():
    if entity not in CHEXPERT_COLS:
        continue
    positives = eval_df[eval_df[entity] == 1.0]
    negatives = eval_df[eval_df[entity] == 0.0]

    def _mentioned(report, kws=keywords):
        r = str(report).lower()
        return any(kw in r for kw in kws)

    hit_rate   = positives['generated_report'].apply(_mentioned).mean() if len(positives) > 0 else float('nan')
    false_rate = negatives['generated_report'].apply(_mentioned).mean() if len(negatives) > 0 else float('nan')

    results_by_pathology[entity] = {
        'n_positive':        len(positives),
        'n_negative':        len(negatives),
        'mention_rate':      hit_rate,
        'false_mention_rate':false_rate,
    }


assert results_by_pathology, (
    f"STOP: results_by_pathology is empty.\n"
    f"  CHEXPERT_COLS: {CHEXPERT_COLS[:5]}\n"
    "  None of the CHEXPERT_REPORT_KEYWORDS keys matched CHEXPERT_COLS. "
    "Check chexpert.csv column names."
)
assert any(v['n_positive'] > 0 for v in results_by_pathology.values()), (
    f"STOP: Every pathology has n_positive=0 — CheXpert labels are all NaN.\n"
    f"  test_subset study_id sample: {test_subset['study_id'].iloc[:3].tolist()}\n"
    f"  chexpert_df study_id sample: {chexpert_df['study_id'].iloc[:3].tolist()}\n"
    "  CheXpert study_id format mismatch — re-check Cell 2 normalisation."
)

print("=" * 72)
print(f"PATHOLOGY ALIGNMENT METRICS — R2Gen on MIMIC-CXR ({len(baseline_df)} tests)")
print("=" * 72)
print(f"  {'Entity':<32} {'N+':>5} {'MentionRate':>12} {'FalseMenRate':>13}")
print("-" * 72)
for ent, r in results_by_pathology.items():
    m  = f"{r['mention_rate']*100:5.1f}%"     if not pd.isna(r['mention_rate'])       else "   N/A"
    fm = f"{r['false_mention_rate']*100:5.1f}%" if not pd.isna(r['false_mention_rate']) else "   N/A"
    print(f"  {ent:<32} {r['n_positive']:>5} {m:>12} {fm:>13}")

valid_mentions = [v['mention_rate'] for v in results_by_pathology.values() if not pd.isna(v['mention_rate'])]
mean_mention   = sum(valid_mentions) / len(valid_mentions) if valid_mentions else 0.0

print(f"\nMean pathology mention rate : {mean_mention*100:.1f}%")
print(f"  (% of CheXpert-confirmed pathologies present in generated reports)")


gen_words  = [str(r).split() for r in baseline_df['report']]
vocab_size = len(set(w for ws in gen_words for w in ws))
avg_len    = sum(len(ws) for ws in gen_words) / max(1, len(gen_words))
unique_rt  = baseline_df['report'].nunique() / max(1, len(baseline_df))

print(f"\nLexical statistics:")
print(f"  Vocabulary size :  {vocab_size}")
print(f"  Avg report length: {avg_len:.1f} words")
print(f"  Unique report %  : {unique_rt*100:.1f}%")

print("\n--- Sample outputs ---")
for i in range(min(3, len(baseline_df))):
    row = baseline_df.iloc[i]
    chex_pos = [c for c in CHEXPERT_COLS if eval_df.iloc[i].get(c) == 1.0] if i < len(eval_df) else []
    print(f"\n[{i+1}] {row['study_id']}")
    print(f"  GEN:    {row['report'][:120]}")
    print(f"  ChEx+:  {chex_pos}")

metrics = {
    'mean_pathology_mention_rate': float(mean_mention),
    'vocabulary_size':             int(vocab_size),
    'avg_report_length':           float(avg_len),
    'unique_report_rate':          float(unique_rt),
    'note': 'BLEU/ROUGE not computed: GT radiology text not available in MIMIC-CXR-JPG dataset',
}
metrics_path = os.path.join(OUTPUT_DIR, 'baseline_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\n✅ Saved: {metrics_path}")

In [ ]:
test_info = pd.DataFrame([{
    'study_id': str(row['study_id']),
    'image_path': get_best_image(row) or "",
    'ground_truth': get_report_text(row),
} for _, row in test_subset.iterrows()])

test_info.to_csv(os.path.join(OUTPUT_DIR, 'test_samples.csv'), index=False)


zip_path = '/kaggle/working/module2_outputs'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)

print("="*60)
print("MODULE 2 COMPLETE — R2Gen on MIMIC-CXR")
print("="*60)
print(f)

In [ ]:
from abc import ABC, abstractmethod
from typing import List, Dict, Optional
from pathlib import Path

class BaseGeneratorAdapter(ABC):
    
    name: str = "base"

    @abstractmethod
    def load_model(self, checkpoint_path: str, device: str = "cuda") -> None:
        
        ...

    @abstractmethod
    def generate(
        self,
        image_paths: List[str],
        batch_size: int = 16,
    ) -> List[Dict[str, str]]:
        
        ...

    def save_predictions(self, predictions: List[Dict], output_path: str) -> None:
        
        df = pd.DataFrame(predictions)
        df.to_csv(output_path, index=False)
        print(f"✅ Saved {len(df)} predictions to {output_path}")


class R2GenAdapter(BaseGeneratorAdapter):
    
    name = "r2gen"

    def __init__(self, model, tokenizer, transform, device):
        self.model = model
        self.tokenizer = tokenizer
        self.transform = transform
        self.device = device

    def load_model(self, checkpoint_path: str, device: str = "cuda") -> None:
        
        pass

    def generate(self, image_paths: List[str], batch_size: int = 16) -> List[Dict]:
        results = []
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i + batch_size]
            batch_imgs = []
            for p in batch_paths:
                try:
                    img = Image.open(p).convert('RGB')
                    batch_imgs.append(self.transform(img))  
                except Exception:
                    batch_imgs.append(torch.zeros(3, 224, 224))  
            images = torch.stack(batch_imgs).to(self.device)  
            with torch.no_grad():
                output = self.model(images, mode='sample')
                reports = self.tokenizer.decode_batch(output.cpu().numpy())
            for r in reports:
                results.append({'report': r.strip() if r else '', 'log_prob': 0.0})
        return results


class R2GenCMNAdapter(BaseGeneratorAdapter):
    
    name = "r2gencmn"

    def __init__(self):
        self.model = None
        self.tokenizer = None

    def load_model(self, checkpoint_path: str, device: str = "cuda") -> None:
        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(
                f"R2GenCMN checkpoint not found: {checkpoint_path}\n"
                "Download from: https://github.com/zhjohnchan/R2GenCMN"
            )
        
        
        
        
        raise NotImplementedError("R2GenCMN loading not yet implemented — checkpoint required")

    def generate(self, image_paths: List[str], batch_size: int = 16) -> List[Dict]:
        if self.model is None:
            raise RuntimeError("R2GenCMN model not loaded. Call load_model() first.")
        raise NotImplementedError("R2GenCMN generation not yet implemented")



GENERATORS = {}


r2gen_adapter = R2GenAdapter(model, wrapped_tok, eval_transform, DEVICE)
GENERATORS['r2gen'] = r2gen_adapter
print(f"✅ Registered generator: r2gen")


R2GENCMN_CHECKPOINT = "/kaggle/working/model_mimic_cxr_r2gencmn.pth"
if os.path.exists(R2GENCMN_CHECKPOINT):
    try:
        r2gencmn_adapter = R2GenCMNAdapter()
        r2gencmn_adapter.load_model(R2GENCMN_CHECKPOINT, str(DEVICE))
        GENERATORS['r2gencmn'] = r2gencmn_adapter
        print(f"✅ Registered generator: r2gencmn")
    except (FileNotFoundError, NotImplementedError) as e:
        print(f"⚠️ R2GenCMN not available: {e}")
else:
    print(f"⚠️ R2GenCMN checkpoint not found at {R2GENCMN_CHECKPOINT}")
    print(f"   Paper must document single-generator limitation (CLAIM 2024)")

print(f"\nAvailable generators: {list(GENERATORS.keys())}")
print(f"Active for Module 4: r2gen (predictions_baseline.csv already saved)")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import re as _re
from collections import Counter as _Counter

IEEE_SINGLE_COL = 3.5
IEEE_DOUBLE_COL = 7.16

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'font.size': 8, 'axes.titlesize': 9, 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.5,
    'grid.linestyle': '--', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.6, 'lines.linewidth': 1.2, 'patch.linewidth': 0.5,
})

C_BLUE   = '
C_RED    = '
C_GREEN  = '
C_ORANGE = '
C_GRAY   = '

FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)




fig, ax = plt.subplots(figsize=(IEEE_DOUBLE_COL, 3.5))

ents   = [e for e in results_by_pathology if results_by_pathology[e]['n_positive'] >= 5]
mr     = [results_by_pathology[e]['mention_rate'] * 100 for e in ents]
fmr    = [results_by_pathology[e]['false_mention_rate'] * 100 for e in ents]
x      = np.arange(len(ents))
w      = 0.35

bars1 = ax.bar(x - w/2, mr,  w, color=C_BLUE,  label='Mention Rate (CheXpert+)', edgecolor='white', lw=0.5)
bars2 = ax.bar(x + w/2, fmr, w, color=C_RED,   label='False Mention Rate (CheXpert−)', edgecolor='white', lw=0.5)
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.0f}%', ha='center', va='bottom', fontsize=5.5)

ax.set_xticks(x)
ax.set_xticklabels([e.replace(' ', '\n') for e in ents], fontsize=6)
ax.set_ylabel('Rate (%)')
ax.set_title('CheXpert Pathology Alignment — R2Gen Generated Reports (MIMIC-CXR)')
ax.legend(loc='upper right', framealpha=0.9)
ax.set_ylim([0, max(max(mr, default=0), max(fmr, default=0)) + 12])

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m2_chexpert_alignment.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m2_chexpert_alignment.pdf'))
plt.show()
print("✅ Saved: fig_m2_chexpert_alignment.{png,pdf}")




fig, ax = plt.subplots(figsize=(IEEE_SINGLE_COL, 2.8))

gen_lengths = baseline_df['report'].apply(lambda x: len(str(x).split()))
bins_len = np.arange(0, gen_lengths.max() + 5, 5)
ax.hist(gen_lengths, bins=bins_len, color=C_BLUE, alpha=0.8,
        label=f'Generated (μ={gen_lengths.mean():.1f}, σ={gen_lengths.std():.1f})',
        edgecolor='white', linewidth=0.3)
ax.axvline(gen_lengths.mean(), color=C_RED, linestyle='--', lw=1.0, alpha=0.8)

ax.set_xlabel('Report Length (words)')
ax.set_ylabel('Count')
ax.set_title('Generated Report Length Distribution (MIMIC-CXR)')
ax.legend(framealpha=0.9)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m2_length_distribution.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m2_length_distribution.pdf'))
plt.show()
print("✅ Saved: fig_m2_length_distribution.{png,pdf}")




fig, ax = plt.subplots(figsize=(IEEE_DOUBLE_COL, 4.0))

CLINICAL_TERMS = {
    'heart','cardiac','cardiomegaly','lung','lungs','pulmonary',
    'pleural','effusion','opacity','consolidation','atelectasis',
    'pneumothorax','edema','normal','clear','stable','unchanged',
    'mediastinal','mediastinum','aorta','aortic','diaphragm',
    'hilar','bilateral','fracture','nodule','mass','infiltrate',
    'catheter','pacemaker','sternotomy','calcification',
    'upper','lower','right','left','base','apex','lobe',
    'rib','spine','thoracic','bony','osseous',
}

gen_counter = _Counter()
for report in baseline_df['report']:
    for w in _re.findall(r'\b\w+\b', str(report).lower()):
        if w in CLINICAL_TERMS:
            gen_counter[w] += 1

top_terms = [t for t, _ in gen_counter.most_common(20)]
gen_vals  = [gen_counter[t] for t in top_terms]
y = np.arange(len(top_terms))

ax.barh(y, gen_vals, color=C_BLUE, label='Generated', edgecolor='white', lw=0.3)
ax.set_yticks(y)
ax.set_yticklabels(top_terms)
ax.set_xlabel('Frequency')
ax.set_title('Top-20 Clinical Terms in Generated Reports (MIMIC-CXR)')
ax.legend(loc='lower right', framealpha=0.9)
ax.invert_yaxis()

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m2_clinical_terms.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m2_clinical_terms.pdf'))
plt.show()
print("✅ Saved: fig_m2_clinical_terms.{png,pdf}")




fig, ax = plt.subplots(figsize=(IEEE_DOUBLE_COL, 3.0))

path_labels = [c for c in CHEXPERT_COLS if c != 'No Finding']
pos_counts  = [(test_subset[c] == 1.0).sum() for c in path_labels]
unc_counts  = [(test_subset[c] == -1.0).sum() for c in path_labels]
total       = len(test_subset)

x  = np.arange(len(path_labels))
w  = 0.4

b1 = ax.bar(x - w/2, [v/total*100 for v in pos_counts], w,
            color=C_BLUE, label='Positive (CheXpert=1)', edgecolor='white', lw=0.5)
b2 = ax.bar(x + w/2, [v/total*100 for v in unc_counts], w,
            color=C_ORANGE, label='Uncertain (CheXpert=−1)', edgecolor='white', lw=0.5)

ax.set_xticks(x)
ax.set_xticklabels([l.replace(' ', '\n') for l in path_labels], fontsize=6)
ax.set_ylabel('Prevalence in Test Set (%)')
ax.set_title('CheXpert Pathology Distribution — Test Set (MIMIC-CXR Chunk 01)')
ax.legend(loc='upper right', framealpha=0.9)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m2_pathology_prevalence.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m2_pathology_prevalence.pdf'))
plt.show()
print("✅ Saved: fig_m2_pathology_prevalence.{png,pdf}")




fig, ax = plt.subplots(figsize=(IEEE_DOUBLE_COL, 2.0))
ax.axis('off')

def _pct_pa(df):
    return f"{(df['ViewPosition'] == 'PA').sum() / max(1, len(df)) * 100:.0f}%"

table_data = [
    ['Split', 'Studies', 'PA views%', 'With Image', 'CheXpert+≥1'],
    ['Train',
     f"{len(train_df):,}",
     _pct_pa(train_df),
     f"{sum(1 for _, r in train_df.iterrows() if get_best_image(r)):,}",
     f"{((train_df[CHEXPERT_COLS] == 1.0).any(axis=1)).sum():,}"],
    ['Validate',
     f"{len(val_subset):,}",
     _pct_pa(val_subset),
     f"{sum(1 for _, r in val_subset.iterrows() if get_best_image(r)):,}",
     f"{((val_subset[CHEXPERT_COLS] == 1.0).any(axis=1)).sum():,}"],
    ['Test',
     f"{len(test_subset):,}",
     _pct_pa(test_subset),
     f"{sum(1 for _, r in test_subset.iterrows() if get_best_image(r)):,}",
     f"{((test_subset[CHEXPERT_COLS] == 1.0).any(axis=1)).sum():,}"],
]

table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.0, 1.5)
for j in range(len(table_data[0])):
    table[0, j].set_facecolor('
    table[0, j].set_text_props(color='white', weight='bold')

ax.set_title('MIMIC-CXR Chunk 01 Dataset Statistics', fontsize=9, pad=10)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_m2_dataset_stats.png'))
fig.savefig(os.path.join(FIG_DIR, 'fig_m2_dataset_stats.pdf'))
plt.show()
print("✅ Saved: fig_m2_dataset_stats.{png,pdf}")

print("\n" + "=" * 60)
print("MODULE 2 — PUBLICATION FIGURES SUMMARY")
print("=" * 60)
print(f)